# EAGLE3 Per-Step Latency Decomposition

Analyze output of `scripts/sweep_eagle3_latency.py` — per-step cost broken into
`target_forward`, `eagle3_draft`, `verify_greedy`, `verify_overhead`, `post_verify`.

Oracle patch forces `accept_length = 0`, so `step_ms` is the raw cost of one speculative
iteration with **no acceptance benefit** — use this to characterize the tree-size cost curve,
not end-to-end speedup.

Input: `{BASE}/eagle3_sweep3d.json`

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.rcParams.update({
    'font.size': 11,
    'figure.figsize': (12, 5),
    'figure.dpi': 100,
})

In [ ]:
BENCH_PATH = "../results/qwen3_14b/eagle3_sweep3d.json"

with open(BENCH_PATH) as f:
    data = json.load(f)

vanilla_tpot = data['vanilla_tpot_ms']
all_df = pd.DataFrame(data['results'])
ok = all_df[all_df.get('error').isna()].reset_index(drop=True) if 'error' in all_df.columns else all_df.copy()
err = all_df[all_df.get('error').notna()].reset_index(drop=True) if 'error' in all_df.columns else pd.DataFrame()

print(f"Model: {data['model']}")
print(f"Draft: {data['draft_model']}")
print(f"Vanilla step_ms: {vanilla_tpot:.2f}")
print(f"Configs: {len(all_df)} total, {len(ok)} ok, {len(err)} errored")
print(f"Grid: topks={sorted(ok['topk'].unique())}, steps={sorted(ok['steps'].unique())}, budgets={sorted(ok['budget'].unique())}")

## Summary table

All successful configs with their decomposition. `overhead_ms = step_ms - target_forward_ms - eagle3_draft_ms` captures verify + post_verify + misc.

In [ ]:
view_cols = ['topk', 'steps', 'budget', 'step_ms', 'target_forward_ms',
             'eagle3_draft_ms', 'verify_overhead_ms', 'post_verify_ms',
             'verify_greedy_ms', 'n_samples']
summary = ok[view_cols].sort_values('step_ms').reset_index(drop=True)
summary.style.background_gradient(subset=['step_ms'], cmap='RdYlGn_r') \
             .background_gradient(subset=['verify_overhead_ms'], cmap='Reds') \
             .format({c: '{:.2f}' for c in view_cols if c.endswith('_ms')})

## Stacked decomposition across all configs

Each bar is one config sorted by step_ms. Components stacked: target forward → EAGLE3 draft → verify overhead → post verify. The dashed line marks vanilla step latency.

In [ ]:
order = ok.sort_values('step_ms').reset_index(drop=True)
labels = [f"k={r.topk} s={r.steps} B={r.budget}" for r in order.itertuples()]

comps = [
    ('target_forward_ms', '#4c72b0', 'target forward'),
    ('eagle3_draft_ms',   '#55a868', 'EAGLE3 draft'),
    ('verify_overhead_ms','#c44e52', 'verify overhead'),
    ('post_verify_ms',    '#8172b2', 'post verify'),
]

fig, ax = plt.subplots(figsize=(max(14, 0.15 * len(order)), 6))
bottom = np.zeros(len(order))
for col, color, label in comps:
    ax.bar(range(len(order)), order[col], bottom=bottom, color=color, label=label, width=0.9)
    bottom = bottom + order[col].values

ax.axhline(vanilla_tpot, ls='--', color='black', alpha=0.6, label=f'vanilla ({vanilla_tpot:.1f} ms)')
ax.set_xticks(range(len(order)))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_ylabel('ms per step')
ax.set_title('EAGLE3 per-step latency decomposition (sorted by step_ms)')
ax.legend(loc='upper left')
ax.margins(x=0.005)
plt.tight_layout()
plt.show()

## Budget sweep per (topk, steps)

For each (topk, steps) combination with ≥2 successful budgets, plot step_ms and its components vs budget. Shows where verify overhead starts dominating.

In [ ]:
pairs = [(t, s) for (t, s), g in ok.groupby(['topk', 'steps']) if len(g) >= 2]
pairs.sort()

ncols = 4
nrows = int(np.ceil(len(pairs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows),
                         sharey=False, squeeze=False)

for ax, (tk, st) in zip(axes.flat, pairs):
    g = ok[(ok.topk == tk) & (ok.steps == st)].sort_values('budget')
    x = g['budget'].values
    ax.plot(x, g['step_ms'], 'o-', color='black', lw=2, label='step (total)')
    ax.plot(x, g['target_forward_ms'], 's--', color='#4c72b0', label='target fwd')
    ax.plot(x, g['eagle3_draft_ms'], 'd--', color='#55a868', label='draft')
    ax.plot(x, g['verify_overhead_ms'], 'v--', color='#c44e52', label='verify ovhd')
    ax.plot(x, g['post_verify_ms'], '^--', color='#8172b2', label='post verify')
    ax.axhline(vanilla_tpot, ls=':', color='gray', alpha=0.7)
    ax.set_title(f'topk={tk}, steps={st}')
    ax.set_xlabel('budget')
    ax.set_ylabel('ms')
    ax.set_xscale('log', base=2)
    if len(x) > 1 and x.max() > 0 and g['step_ms'].max() / g['step_ms'].min() > 3:
        ax.set_yscale('log')
    ax.grid(alpha=0.3)

for ax in axes.flat[len(pairs):]:
    ax.set_visible(False)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Per-step latency vs budget, faceted by (topk, steps)', y=1.0)
plt.tight_layout()
plt.show()

## Component isolation: how each knob affects each cost

1. **Draft cost** should scale with `steps` (draft model runs once per step).
2. **Target forward** should be nearly independent of all knobs (fixed by target model on fixed extend length).
3. **Verify overhead** should scale with `budget` (bigger tree → bigger mask → more work on GPU).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. draft vs steps, colored by topk (avg over budget)
ax = axes[0]
for tk, g in ok.groupby('topk'):
    agg = g.groupby('steps')['eagle3_draft_ms'].mean().reset_index()
    ax.plot(agg['steps'], agg['eagle3_draft_ms'], 'o-', label=f'topk={tk}')
ax.set_xlabel('steps')
ax.set_ylabel('eagle3_draft_ms (avg over budget)')
ax.set_title('Draft cost vs steps')
ax.legend()
ax.grid(alpha=0.3)

# 2. target_forward distribution across all configs
ax = axes[1]
ax.hist(ok['target_forward_ms'], bins=30, color='#4c72b0', edgecolor='white')
ax.axvline(ok['target_forward_ms'].median(), color='black', ls='--',
           label=f'median={ok["target_forward_ms"].median():.2f}')
ax.set_xlabel('target_forward_ms')
ax.set_ylabel('count')
ax.set_title('Target forward distribution (should be narrow)')
ax.legend()

# 3. verify_overhead vs budget, separate line per (topk, steps)
ax = axes[2]
for (tk, st), g in ok.groupby(['topk', 'steps']):
    g = g.sort_values('budget')
    if len(g) >= 2:
        ax.plot(g['budget'], g['verify_overhead_ms'], 'o-', alpha=0.6, lw=1,
                label=f'k={tk} s={st}')
ax.set_xlabel('budget')
ax.set_ylabel('verify_overhead_ms')
ax.set_title('Verify overhead vs budget')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(alpha=0.3, which='both')
# too many lines for a full legend — show only once outside
ax.legend(fontsize=6, ncol=2, loc='upper left')

plt.tight_layout()
plt.show()

## Heatmaps: step_ms over (topk, steps) at fixed budgets

One heatmap per budget value. Missing cells = error configs (budget exceeded SGLang `score_pool`).

In [ ]:
budgets_to_show = [1, 4, 16, 64, 256, 1024]
budgets_to_show = [b for b in budgets_to_show if b in ok['budget'].unique()]

ncols = min(3, len(budgets_to_show))
nrows = int(np.ceil(len(budgets_to_show) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.6 * nrows), squeeze=False)

vmin = ok['step_ms'].min()
vmax = ok['step_ms'].max()

for ax, B in zip(axes.flat, budgets_to_show):
    sub = ok[ok['budget'] == B]
    pivot = sub.pivot(index='steps', columns='topk', values='step_ms')
    pivot = pivot.sort_index().sort_index(axis=1)
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
                   norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax))
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('topk')
    ax.set_ylabel('steps')
    ax.set_title(f'step_ms @ budget={B}')
    for (i, j), v in np.ndenumerate(pivot.values):
        if not np.isnan(v):
            ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                    color='white' if v > (vmin * vmax) ** 0.5 else 'black', fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)

for ax in axes.flat[len(budgets_to_show):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## Failed configs (for reference)

These hit `RuntimeError: selected index k out of range` in SGLang's `organize_draft_results`.
The score pool size is `topk + (steps-1) * topk²`, and budget must satisfy `budget ≤ score_pool + 1`.

In [ ]:
if len(err) > 0:
    err_view = err[['topk', 'steps', 'budget', 'max_capacity']].copy()
    err_view['score_pool'] = err_view['topk'] + (err_view['steps'] - 1) * err_view['topk'] ** 2
    err_view['budget_limit'] = err_view['score_pool'] + 1
    err_view['over_by'] = err_view['budget'] - err_view['budget_limit']
    print(f'{len(err_view)} failed configs:')
    print(err_view.to_string(index=False))
else:
    print('No failed configs.')